In [1]:
import torch

if not torch.cuda.is_available():
    print("CUDA is not available")
else:
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {gpu.total_memory / 1024**3:.1f} GiB")

!nvidia-smi

GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GiB
Wed Oct 21 03:00:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             45W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----

In [2]:
!pip install -U transformers accelerate

In [3]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

model_id = "Qwen/Qwen3-VL-8B-Instruct"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Loaded:", model_id)
print("Weight dtype:", next(model.parameters()).dtype)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Loaded: Qwen/Qwen3-VL-8B-Instruct
Weight dtype: torch.bfloat16


In [4]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!find /content/drive -name "cohort_ocr_v2.csv" -print 2>/dev/null

/content/drive/MyDrive/cohort_ocr_v2.csv
/content/drive/MyDrive/Lighthouse F26/cohort_ocr_v2.csv


In [6]:
import pandas as pd

csv_path = "/content/drive/MyDrive/cohort_ocr_v2.csv"
df = pd.read_csv(csv_path)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head(3))

Rows: 180
Columns: ['image', 'collector', 'platform', 'label', 'all_labels', 'justification', 'text', 'text_old']


,image,collector,platform,label,all_labels,justification,text,text_old
0,Arjun_01.png,Arjun,web,preselection,preselection,Pop up to add items to your order that You wer...,Back to store Complete your order Delivery det...,x Complete your order Dr Pepper $2.70 Popular ...
1,Arjun_02.png,Arjun,web,hidden_information,hidden_information;preselection,Promises a price and discount throughout the w...,Deiveiy Oplons Send asa gift Priority Add a pe...,Meivery Veils i Send asa gift > ae Add a perso...
2,Arjun_03.png,Arjun,web,urgency,urgency,Coundown timer on a sale that was there yester...,Free Shipping by Amazon Get FREE Shipping on e...,{-) Free Shipping by Amazon Get FREE Shipping ...


In [7]:
import os

print("My Drive folders/files:")
print(os.listdir("/content/drive/MyDrive")[:50])

!find /content/drive/MyDrive -type f -name "Arjun_01.png" -print 2>/dev/null

My Drive folders/files:
['Colab Notebooks', 'cohort_ocr_v2.csv', 'Lighthouse F26']
/content/drive/MyDrive/Lighthouse F26/Arjun/Arjun_01.png


In [8]:
import os

image_root = "/content/drive/MyDrive/Lighthouse F26"
image_paths = {}

for root, folders, files in os.walk(image_root):
    for filename in files:
        if filename.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
            image_paths[filename] = os.path.join(root, filename)

missing = [name for name in df["image"].astype(str) if name not in image_paths]

print("Images found:", len(image_paths))
print("CSV rows without a matching image:", len(missing))
print("Missing examples:", missing[:10])
print("Example path:", image_paths.get(df.iloc[0]["image"]))

Images found: 197
CSV rows without a matching image: 0
Missing examples: []
Example path: /content/drive/MyDrive/Lighthouse F26/Arjun/Arjun_01.png


In [9]:
!pip install -q qwen-vl-utils

In [13]:
import json
from pathlib import Path
from qwen_vl_utils import process_vision_info

RATING_PROMPT = """
Classify the screenshot using only visible evidence. Do not speculate about
content below the viewport, later interactions, factual truth, or hidden behavior.
Use the OCR text as a reading aid; base judgments on what is visibly shown.

Rate each category from 0 to 3:
0 = no qualifying sign; 1 = weak sign; 2 = clear sign; 3 = strong, obvious sign.

Categories:
- confirmshaming: The decline option uses guilt, shame, negative identity, or
  loss-framed wording to pressure acceptance. Neutral "No thanks" or "Skip"
  does not qualify.
- hidden_information: A material renewal charge, mandatory fee, contract term,
  or meaningful alternative is visibly hard to notice. Compare its visual
  prominence with the main offer. A high price alone does not qualify.
- preselection: An optional seller-favoring choice is visibly selected by
  default. A checked box or enabled toggle can qualify. Consider whether an
  OFF opt-out control leaves the underlying optional activity enabled. A
  recommendation, badge, popup, or plus button alone does not qualify.
- scarcity: Explicit remaining-stock or limited-quantity language, such as
  "only one left" or "almost sold out." A discount, timer, review count, or
  bestseller badge alone does not qualify.
- urgency: An explicit countdown or deadline for an offer or action. Delivery
  estimates and limited quantity alone do not qualify.
- social_proof: Visible purchase counts or other people's purchase, viewing,
  or activity claims used to persuade. Ordinary star ratings or reviews alone
  do not qualify.

Return only valid JSON with exactly these two keys:
{
  "justification": "One or two sentences naming the visible cue, quoting relevant
  words when present, or explaining why no cue qualifies.",
  "ratings": {
    "confirmshaming": 0,
    "hidden_information": 0,
    "preselection": 0,
    "scarcity": 0,
    "urgency": 0,
    "social_proof": 0
  }
}
All six ratings must be integers from 0 to 3. Do not include a predicted label.
"""

def rate_one(row):
    image_name = str(row["image"])
    image_path = image_paths[image_name]
    ocr_text = "" if pd.isna(row["text"]) else str(row["text"])

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": str(Path(image_path).resolve())},
            {"type": "text", "text": RATING_PROMPT + "\n\nOCR text:\n" + ocr_text}
        ]
    }]

    prompt_text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    images, videos, video_kwargs = process_vision_info(
        messages,
        image_patch_size=16,
        return_video_kwargs=True,
        return_video_metadata=True
    )

    if videos is not None:
        videos, video_metadata = zip(*videos)
        videos, video_metadata = list(videos), list(video_metadata)
    else:
        video_metadata = None

    inputs = processor(
        text=[prompt_text],
        images=images,
        videos=videos,
        video_metadata=video_metadata,
        return_tensors="pt",
        do_resize=False,
        **video_kwargs
    ).to(model.device)

    output_ids = model.generate(**inputs, max_new_tokens=300, do_sample=False)
    new_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_ids, skip_special_tokens=True)[0].strip()

In [15]:
import re
import json

sample_df = df.sample(n=10, random_state=42)

for _, row in sample_df.iterrows():
    print(f"\nTesting: {row['image']}")
    try:
        raw = rate_one(row)
        match = re.search(r"\{.*\}", raw, flags=re.DOTALL)

        if not match:
            print("Could not find JSON in model response:")
            print(raw)
            continue

        result = json.loads(match.group(0))
        print(json.dumps(result, indent=2))

    except Exception as e:
        print("Error:", repr(e))


Testing: Khushi_10.png
{
  "justification": "The 'BESTSELLER' badge on the right product card is a visible indicator of popularity, which qualifies as social proof. No other cues for confirmshaming, hidden information, preselection, scarcity, urgency, or other categories are present in the visible content.",
  "ratings": {
    "confirmshaming": 0,
    "hidden_information": 0,
    "preselection": 0,
    "scarcity": 0,
    "urgency": 0,
    "social_proof": 1
  }
}

Testing: Khushi_34.png
{
  "justification": "The popup presents an offer with a 45-day payment window, which is a deadline. The text 'pay us back within 45 days!' explicitly states a time-bound condition, qualifying as urgency. No other categories are visibly supported by the text or layout.",
  "ratings": {
    "confirmshaming": 0,
    "hidden_information": 0,
    "preselection": 0,
    "scarcity": 0,
    "urgency": 2,
    "social_proof": 0
  }
}

Testing: Arjun_23.jpg
{
  "justification": "The popup displays a countdown tim

In [16]:
import re
import json
import pandas as pd

categories = [
    "confirmshaming",
    "hidden_information",
    "preselection",
    "scarcity",
    "urgency",
    "social_proof"
]

results = []
failed = []

for count, (_, row) in enumerate(df.iterrows(), start=1):
    try:
        raw = rate_one(row)
        match = re.search(r"\{.*\}", raw, flags=re.DOTALL)
        if not match:
            raise ValueError("No JSON found in response")

        result = json.loads(match.group(0))
        ratings = result["ratings"]

        if (
            set(result.keys()) != {"justification", "ratings"}
            or not isinstance(result["justification"], str)
            or set(ratings.keys()) != set(categories)
            or any(type(ratings[c]) is not int or not 0 <= ratings[c] <= 3
                   for c in categories)
        ):
            raise ValueError("Response did not match the required JSON format")

        strength = max(ratings.values())
        predicted = (
            next(c for c in categories if ratings[c] == strength)
            if strength > 0 else "none"
        )

        results.append({
            "image": str(row["image"]),
            "label": row["label"],
            "predicted": predicted,
            "justification": result["justification"],
            **ratings,
            "strength": strength
        })

    except Exception as e:
        failed.append(str(row["image"]))
        print(f"Failed: {row['image']} — {e}")

    if count % 10 == 0:
        pd.DataFrame(results).to_csv(
            "/content/drive/MyDrive/ratings_partial.csv", index=False
        )
        print(f"Processed {count}/{len(df)}; successful: {len(results)}")

ratings_df = pd.DataFrame(results, columns=[
    "image", "label", "predicted", "justification",
    *categories, "strength"
])
ratings_df.to_csv("/content/drive/MyDrive/ratings.csv", index=False)

print(f"\nSaved {len(ratings_df)} ratings to MyDrive/ratings.csv")
print("Failed images:", failed)

Processed 10/180; successful: 10
Processed 20/180; successful: 20
Processed 30/180; successful: 30
Processed 40/180; successful: 40
Processed 50/180; successful: 50
Processed 60/180; successful: 60
Processed 70/180; successful: 70
Processed 80/180; successful: 80
Processed 90/180; successful: 90
Processed 100/180; successful: 100
Processed 110/180; successful: 110
Processed 120/180; successful: 120
Processed 130/180; successful: 130
Processed 140/180; successful: 140
Processed 150/180; successful: 150
Processed 160/180; successful: 160
Processed 170/180; successful: 170
Processed 180/180; successful: 180

Saved 180 ratings to MyDrive/ratings.csv
Failed images: []


In [17]:
import re
import json
import pandas as pd

TEXT_ONLY_PROMPT = """
Classify using only the OCR text below. No screenshot is provided.
Do not infer visual prominence, layout, font size, color, or whether a
checkbox or toggle is selected unless the OCR text explicitly says so.
Do not speculate beyond the OCR text.

""" + RATING_PROMPT[RATING_PROMPT.index("Rate each category from 0 to 3:"):]

def rate_text_one(row):
    ocr_text = "" if pd.isna(row["text"]) else str(row["text"])
    messages = [{
        "role": "user",
        "content": [{
            "type": "text",
            "text": TEXT_ONLY_PROMPT + "\n\nOCR text:\n" + ocr_text
        }]
    }]
    prompt_text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=[prompt_text], return_tensors="pt").to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=250, do_sample=False)
    new_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_ids, skip_special_tokens=True)[0].strip()

categories = [
    "confirmshaming", "hidden_information", "preselection",
    "scarcity", "urgency", "social_proof"
]

for _, row in df.sample(n=10, random_state=42).iterrows():
    print(f"\nTesting: {row['image']}")
    try:
        raw = rate_text_one(row)
        match = re.search(r"\{.*\}", raw, flags=re.DOTALL)
        result = json.loads(match.group(0)) if match else None
        valid = (
            isinstance(result, dict)
            and set(result) == {"justification", "ratings"}
            and isinstance(result["justification"], str)
            and set(result["ratings"]) == set(categories)
            and all(type(result["ratings"][c]) is int
                    and 0 <= result["ratings"][c] <= 3
                    for c in categories)
        )
        print("Valid JSON:", valid)
        print(json.dumps(result, indent=2) if valid else raw)
    except Exception as e:
        print("ERROR:", repr(e))


Testing: Khushi_10.png
Valid JSON: True
{
  "justification": "The OCR text contains promotional offers like 'SUBSCRIBE AND SAVE 25%' and 'BESTSELLER', but no explicit guilt, hidden fees, preselected options, scarcity language, urgency countdowns, or social proof claims such as purchase counts. No qualifying cues for any category are present.",
  "ratings": {
    "confirmshaming": 0,
    "hidden_information": 0,
    "preselection": 0,
    "scarcity": 0,
    "urgency": 0,
    "social_proof": 0
  }
}

Testing: Khushi_34.png
Valid JSON: True
{
  "justification": "The OCR text contains no explicit guilt, shame, hidden fees, preselected options, scarcity language, urgency countdowns, or social proof claims. The phrase 'Accept Jved ffers' may imply a default action, but no visible checkbox or toggle is mentioned, and 'Remind me later ?' is not a preselected option. No qualifying cues are present.",
  "ratings": {
    "confirmshaming": 0,
    "hidden_information": 0,
    "preselection": 0,
  

In [18]:
results = []
failed = []

for count, (_, row) in enumerate(df.iterrows(), start=1):
    try:
        raw = rate_text_one(row)
        match = re.search(r"\{.*\}", raw, flags=re.DOTALL)
        if not match:
            raise ValueError("No JSON found")

        result = json.loads(match.group(0))
        ratings = result["ratings"]

        if (
            set(result) != {"justification", "ratings"}
            or not isinstance(result["justification"], str)
            or set(ratings) != set(categories)
            or any(type(ratings[c]) is not int or not 0 <= ratings[c] <= 3
                   for c in categories)
        ):
            raise ValueError("Invalid JSON structure or rating")

        strength = max(ratings.values())
        predicted = (
            next(c for c in categories if ratings[c] == strength)
            if strength > 0 else "none"
        )

        results.append({
            "image": str(row["image"]),
            "label": row["label"],
            "predicted": predicted,
            "justification": result["justification"],
            **ratings,
            "strength": strength
        })

    except Exception as e:
        failed.append(str(row["image"]))
        print(f"Failed: {row['image']} — {e}")

    if count % 10 == 0:
        pd.DataFrame(results).to_csv(
            "/content/drive/MyDrive/text_ratings_partial.csv", index=False
        )
        print(f"Processed {count}/180; successful: {len(results)}")

text_df = pd.DataFrame(results, columns=[
    "image", "label", "predicted", "justification",
    *categories, "strength"
])
text_df.to_csv("/content/drive/MyDrive/text_only.csv", index=False)
text_df.to_csv("/content/drive/MyDrive/text_ratings.csv", index=False)

print(f"Saved {len(text_df)} text-only ratings.")
print("Failed images:", failed)

Processed 10/180; successful: 10
Processed 20/180; successful: 20
Processed 30/180; successful: 30
Processed 40/180; successful: 40
Processed 50/180; successful: 50
Processed 60/180; successful: 60
Processed 70/180; successful: 70
Processed 80/180; successful: 80
Processed 90/180; successful: 90
Processed 100/180; successful: 100
Processed 110/180; successful: 110
Processed 120/180; successful: 120
Processed 130/180; successful: 130
Processed 140/180; successful: 140
Processed 150/180; successful: 150
Processed 160/180; successful: 160
Processed 170/180; successful: 170
Processed 180/180; successful: 180
Saved 180 text-only ratings.
Failed images: []


In [19]:
text_df.to_csv("/content/drive/MyDrive/text_only.csv", index=False)
text_df.to_csv("/content/drive/MyDrive/text_ratings.csv", index=False)

print(f"Saved {len(text_df)} rows to Google Drive.")

Saved 180 rows to Google Drive.
